## Imports

In [ ]:
import os
import shutil
import gc
import random
import numpy as np
import pandas as pd
import dotenv

from datasets import load_dataset
from huggingface_hub import snapshot_download
from pathlib import Path
from torchvision import transforms
import torchvision
from PIL import Image

from tqdm import tqdm
from torchvision.utils import save_image

## Loading & creating env variables

In [ ]:
dotenv.load_dotenv()

HF_DATASET = os.getenv("HF_DATASET_NAME")
HF_TOKEN = os.getenv("HF_TOKEN")
HF_DATASET_LINK = os.getenv("HF_DATASET_LINK")

HF_SPLIT = "train"

LOCAL_DOWNLOAD_DIR = "./originalImages"

## Downloading HuggingFace dataset

In [ ]:
dfOriginal = pd.read_parquet(HF_DATASET_LINK)
df = dfOriginal[["cirenId","totalDeltaVKph"]]
df = df.drop_duplicates(keep='first')  
nulls = df["totalDeltaVKph"].isna()
missing_cirenIds = df.loc[nulls, "cirenId"].tolist()

imgIgnore = [f"**/CIREN/{ciren_id}/*" for ciren_id in missing_cirenIds]
patternIgnore = [".git*", "README.md"] + imgIgnore

In [ ]:

print("Descargando dataset completo...")

snapshot_download(
        repo_id=HF_DATASET,
        repo_type="dataset",
        local_dir=LOCAL_DOWNLOAD_DIR,
        token=HF_TOKEN,
        ignore_patterns= patternIgnore
    )

print("Descarga completada")

## Creating transformed dataset

In [ ]:
TRAIN_DATA_PATH = "./"+LOCAL_DOWNLOAD_DIR+"/images/CIREN/"
OUTPUT_DATA_PATH = Path("trasformedImages/imagenes/CIREN")
TRANSFORM_IMG = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor()  # Convierte a Tensor [0, 1], necesario para save_image
])

raw_image_dataset = torchvision.datasets.ImageFolder(root=TRAIN_DATA_PATH, transform=None)
classes = raw_image_dataset.classes
N_PHOTOSTRANSFORMED = 3
print(len(raw_image_dataset))

# Cantidad FINAL deseada
TARGET_IMAGES = len(raw_image_dataset)-1

In [ ]:
for img_path_str, class_idx in raw_image_dataset.samples:
    img_path = Path(img_path_str)
    class_name = classes[class_idx]
    target_folder = OUTPUT_DATA_PATH / class_name
    target_folder.mkdir(parents=True, exist_ok=True)
    
    img = Image.open(img_path).convert('RGB')
    
    # Generar múltiples variaciones aleatorias para la misma imagen
    for i in range(N_PHOTOSTRANSFORMED):
        transformed_tensor = TRANSFORM_IMG(img)
        # El nombre incluirá el número de la copia: aug_0_foto.jpg, aug_1_foto.jpg...
        output_img_path = target_folder / f"aug_{i}_{img_path.name}"
        save_image(transformed_tensor, output_img_path)

## Add noise to totalDeltaVKph

In [ ]:
noise_percentage = 0.05
noise_factor = np.random.normal(1.0, noise_percentage, size=len(df))

df["totalDeltaVKph"] = df["totalDeltaVKph"] * noise_factor

df["totalDeltaVKph"] = df["totalDeltaVKph"].clip(lower=0)
noise_map = df.set_index('cirenId')['totalDeltaVKph']
dfOriginal['totalDeltaVKph'] = dfOriginal['cirenId'].map(noise_map)
print(dfOriginal[["cirenId", "totalDeltaVKph"]].head(30))

## TODO: Creating parquets